In [ ]:
from pyspark import SparkConf, SparkContext
import matplotlib.pyplot as plt
import os
from datetime import datetime

In [ ]:
def load_file(sc, path):
    """Read CSV file using RDD and strip header"""
    rdd = sc.textFile(path)
    header = rdd.first()
    data = rdd.filter(lambda x: x != header).map(lambda x: x.split(","))
    return data

In [ ]:
def plot_chart(x, y, title, xlabel, ylabel, save_path, chart_type="bar"):
    """Draw a bar or line chart and save it."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.figure(figsize=(12, 6))

    if chart_type == "bar":
        plt.bar(x, y, color="skyblue")
    elif chart_type == "line":
        plt.plot(x, y, marker="o", color="orange")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

In [ ]:
def f1(data):
    result = (data.map(lambda x: (x[2], 1))
                   .reduceByKey(lambda a, b: a + b)
                   .sortBy(lambda x: x[1], ascending=False)
                   .take(100))

    items, counts = zip(*result)
    plot_chart(items, counts,
               "Top 100 most purchased products",
               "Product name", "Number of purchases", "output/f1_top100.png")
    return result

In [ ]:
def f2(data):
    result = (data.map(lambda x: (x[0], x[1]))  # (Member_number, Date)
                   .distinct()
                   .map(lambda x: (x[0], 1))
                   .reduceByKey(lambda a, b: a + b)
                   .sortBy(lambda x: x[1], ascending=False)
                   .take(100))

    customers, baskets = zip(*result)
    plot_chart(customers, baskets,
               "Top 100 customers with the most shopping carts",
               "Customer code", "Cart quantity", "output/f2_top100_customers.png")
    return result

In [ ]:
def f3(data, product_name):
    result = (data.filter(lambda x: x[2].lower() == product_name.lower())
                   .map(lambda x: (datetime.strptime(x[1], "%d/%m/%Y").strftime("%Y-%m"), 1))
                   .reduceByKey(lambda a, b: a + b)
                   .sortByKey()
                   .collect())

    months, counts = zip(*result)
    plot_chart(months, counts,
               f"Amount of product purchases '{product_name}' by month",
               "Month", "Purchase quantity", "output/f3_product_trend.png", chart_type="line")
    return result

In [ ]:
def f4(data, customer_id):
    result = (data.filter(lambda x: x[0] == str(customer_id))
                   .map(lambda x: (x[0], x[1]))
                   .distinct()
                   .map(lambda x: (datetime.strptime(x[1], "%d/%m/%Y").strftime("%Y-%m"), 1))
                   .reduceByKey(lambda a, b: a + b)
                   .sortByKey()
                   .collect())

    months, counts = zip(*result)
    plot_chart(months, counts,
               f"Number of customer baskets per month {customer_id}",
               "Month", "Cart number", "output/f4_customer_trend.png", chart_type="line")
    return result

In [ ]:
def main():
    conf = SparkConf().setAppName("Baskets Analysis").setMaster("local[*]")
    sc = SparkContext(conf=conf)
    path = "baskets.csv"

    print("Reading file...")
    data = load_file(sc, path)

    print("F1: Top selling product")
    f1_result = f1(data)
    print(f1_result[:5])

    print("F2: Top customers with the most shopping carts")
    f2_result = f2(data)
    print(f2_result[:5])

    print("F3: Monthly purchase volume of 'coffee' product")
    f3_result = f3(data, "coffee")
    print(f3_result)

    print("F4: Number of customer shopping carts per month 1249")
    f4_result = f4(data, "1249")
    print(f4_result)

    sc.stop()

In [ ]:
  if __name__ == "__main__":
    main()